# 📊 EDA — Bridge Crack Surface Defect Dataset
**Project:** Manufacturing Defect Detection (Vision)  
**Team:** Baguio · Doton · Bernarte · De Castro  
**Institution:** Holy Angel University, Angeles City  
**Week 2 Deliverable — Exploratory Data Analysis**

---

This notebook performs a full exploratory data analysis on the Bridge Cracks Image dataset sourced from Kaggle (`yidazhang07/bridge-cracks-image`). We examine class distributions, image statistics, sample visualizations, and prepare train/val/test splits for downstream CNN training.

## Table of Contents
1. [Environment Setup](#setup)
2. [Dataset Loading & Directory Audit](#loading)
3. [Class Distribution](#class-dist)
4. [Image Dimension Statistics](#dimensions)
5. [Pixel Intensity Analysis](#intensity)
6. [Sample Image Grid](#samples)
7. [Train / Val / Test Split](#splits)
8. [Summary & Observations](#summary)

## 1. Environment Setup <a id='setup'></a>

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import random
import warnings
from pathlib import Path
from collections import Counter

# ── Data / numerics ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Image handling ────────────────────────────────────────────────────────────
from PIL import Image
import cv2                      # OpenCV for pixel-level stats

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── ML utilities ──────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
random.seed(42)
np.random.seed(42)

print('All imports OK.')

In [ ]:
# ── Configure dataset root ────────────────────────────────────────────────────
# Expected structure after Kaggle download:
#   DATA_ROOT/
#     Crack/          ← images WITH cracks (positive class)
#     Non-Crack/      ← images WITHOUT cracks (negative class)
#
# If running on Colab, mount your Drive or use kaggle API:
#   !kaggle datasets download yidazhang07/bridge-cracks-image --unzip

DATA_ROOT = Path('data/bridge-cracks')          # ← change if needed
CLASSES   = ['Crack', 'Non-Crack']              # folder names = class labels
IMG_EXTS  = {'.jpg', '.jpeg', '.png', '.bmp'}

print(f'Dataset root : {DATA_ROOT.resolve()}')
print(f'Exists       : {DATA_ROOT.exists()}')

## 2. Dataset Loading & Directory Audit <a id='loading'></a>

In [ ]:
# ── Walk the dataset directory and build a master dataframe ───────────────────
records = []

for cls in CLASSES:
    cls_dir = DATA_ROOT / cls
    if not cls_dir.exists():
        print(f'[WARN] Directory not found: {cls_dir}')
        continue
    for fpath in cls_dir.iterdir():
        if fpath.suffix.lower() in IMG_EXTS:
            records.append({'path': str(fpath), 'label': cls})

df = pd.DataFrame(records)
print(f'Total images found : {len(df)}')
print(df['label'].value_counts().to_string())

## 3. Class Distribution <a id='class-dist'></a>

In [ ]:
# ── Bar chart of class counts ─────────────────────────────────────────────────
# An imbalanced dataset would inflate accuracy; we check here so we can
# decide whether to use weighted loss / oversampling in training.

counts = df['label'].value_counts()
imbalance_ratio = counts.max() / counts.min()

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(counts.index, counts.values, color=['#e74c3c', '#2ecc71'], edgecolor='white')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,}', ha='center', va='bottom', fontweight='bold')
ax.set_title('Class Distribution (Bridge Crack Dataset)', fontsize=14)
ax.set_ylabel('Image Count')
ax.set_xlabel('Class')
plt.tight_layout()
plt.savefig('plots/class_distribution.png', dpi=150)
plt.show()
print(f'Imbalance ratio (max/min): {imbalance_ratio:.2f}')
if imbalance_ratio > 1.5:
    print('[NOTE] Class imbalance detected — consider weighted CrossEntropyLoss or oversampling.')

## 4. Image Dimension Statistics <a id='dimensions'></a>

In [ ]:
# ── Sample up to 500 images to measure H×W distributions ─────────────────────
# This informs our resize target for the CNN input layer.
# Sampling avoids long runtimes on large datasets.

sample_n = min(500, len(df))
sampled  = df.sample(sample_n, random_state=42)

widths, heights = [], []
for _, row in sampled.iterrows():
    try:
        img = Image.open(row['path'])
        w, h = img.size           # PIL returns (width, height)
        widths.append(w)
        heights.append(h)
    except Exception as e:
        print(f'[WARN] Could not open {row["path"]}: {e}')

dim_df = pd.DataFrame({'width': widths, 'height': heights})
print(dim_df.describe().round(1))

In [ ]:
# ── Scatter plot: width vs height, coloured by aspect ratio ───────────────────
dim_df['aspect'] = dim_df['width'] / dim_df['height']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(dim_df['width'], dim_df['height'],
                c=dim_df['aspect'], cmap='plasma', alpha=0.5, s=15)
axes[0].set_xlabel('Width (px)')
axes[0].set_ylabel('Height (px)')
axes[0].set_title('Width vs Height')

axes[1].hist(dim_df['aspect'], bins=30, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Aspect Ratio (W/H)')
axes[1].set_ylabel('Count')
axes[1].set_title('Aspect Ratio Distribution')

plt.tight_layout()
plt.savefig('plots/image_dimensions.png', dpi=150)
plt.show()

## 5. Pixel Intensity Analysis <a id='intensity'></a>

In [ ]:
# ── Compute per-channel mean & std for normalization constants ────────────────
# These values will be used as `mean` and `std` in torchvision.transforms.Normalize
# instead of the ImageNet defaults, since our domain differs from ImageNet.

sample_n2 = min(200, len(df))
sampled2  = df.sample(sample_n2, random_state=0)

r_means, g_means, b_means = [], [], []
r_stds,  g_stds,  b_stds  = [], [], []

for _, row in sampled2.iterrows():
    try:
        img = np.array(Image.open(row['path']).convert('RGB'), dtype=np.float32) / 255.0
        r_means.append(img[:,:,0].mean())
        g_means.append(img[:,:,1].mean())
        b_means.append(img[:,:,2].mean())
        r_stds.append(img[:,:,0].std())
        g_stds.append(img[:,:,1].std())
        b_stds.append(img[:,:,2].std())
    except:
        pass

mean_rgb = (np.mean(r_means), np.mean(g_means), np.mean(b_means))
std_rgb  = (np.mean(r_stds),  np.mean(g_stds),  np.mean(b_stds))

print(f'Dataset mean (R,G,B) : {mean_rgb[0]:.4f}, {mean_rgb[1]:.4f}, {mean_rgb[2]:.4f}')
print(f'Dataset std  (R,G,B) : {std_rgb[0]:.4f}, {std_rgb[1]:.4f}, {std_rgb[2]:.4f}')
print()
print('Use these in transforms.Normalize(mean=[...], std=[...]) during training.')

In [ ]:
# ── Overlay histograms per class for a single sample image ────────────────────
# Helps detect if crack images are systematically darker (grey concrete).

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = {'Crack': '#e74c3c', 'Non-Crack': '#2ecc71'}

for cls in CLASSES:
    subset = df[df['label'] == cls].sample(min(50, len(df[df['label'] == cls])), random_state=1)
    all_gray = []
    for _, row in subset.iterrows():
        try:
            gray = np.array(Image.open(row['path']).convert('L'), dtype=np.float32)
            all_gray.append(gray.flatten())
        except:
            pass
    if all_gray:
        merged = np.concatenate(all_gray)
        axes[0].hist(merged, bins=50, alpha=0.5, color=colors[cls], label=cls, density=True)

axes[0].set_title('Grayscale Intensity Distribution by Class')
axes[0].set_xlabel('Pixel Intensity')
axes[0].set_ylabel('Density')
axes[0].legend()

# Mean brightness per class
brightness = []
for _, row in df.sample(min(300, len(df)), random_state=2).iterrows():
    try:
        gray = np.array(Image.open(row['path']).convert('L'), dtype=np.float32)
        brightness.append({'label': row['label'], 'mean_brightness': gray.mean()})
    except:
        pass
bdf = pd.DataFrame(brightness)
sns.boxplot(data=bdf, x='label', y='mean_brightness', ax=axes[1],
            palette=colors)
axes[1].set_title('Mean Brightness per Class')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Mean Pixel Intensity')

plt.tight_layout()
plt.savefig('plots/intensity_analysis.png', dpi=150)
plt.show()

## 6. Sample Image Grid <a id='samples'></a>

In [ ]:
# ── Display 5 samples per class in a grid ─────────────────────────────────────

n_cols = 5
fig, axes = plt.subplots(len(CLASSES), n_cols, figsize=(16, 6))

for row_idx, cls in enumerate(CLASSES):
    subset = df[df['label'] == cls].sample(n_cols, random_state=7)
    for col_idx, (_, item) in enumerate(subset.iterrows()):
        ax = axes[row_idx][col_idx]
        try:
            img = Image.open(item['path']).convert('RGB').resize((224, 224))
            ax.imshow(img)
        except:
            ax.text(0.5, 0.5, 'Error', ha='center', va='center')
        ax.axis('off')
        if col_idx == 0:
            ax.set_ylabel(cls, fontsize=12, rotation=90, labelpad=10)

plt.suptitle('Sample Images — Crack vs Non-Crack', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('plots/sample_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Train / Val / Test Split <a id='splits'></a>

In [ ]:
# ── Stratified 70/15/15 split ─────────────────────────────────────────────────
# Stratification ensures both classes appear proportionally in all three sets.
# We fix random_state=42 for reproducibility.

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15   # must sum to 1.0

# First: split into train and temp (val+test)
train_df, temp_df = train_test_split(
    df,
    test_size=(VAL_RATIO + TEST_RATIO),
    stratify=df['label'],
    random_state=42
)

# Second: split temp into val and test
val_df, test_df = train_test_split(
    temp_df,
    test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
    stratify=temp_df['label'],
    random_state=42
)

# Save split CSVs for reproducible reloading during training
os.makedirs('data/splits', exist_ok=True)
train_df.to_csv('data/splits/train.csv', index=False)
val_df.to_csv('data/splits/val.csv',   index=False)
test_df.to_csv('data/splits/test.csv',  index=False)

print(f'Train : {len(train_df):>6,} images')
print(f'Val   : {len(val_df):>6,} images')
print(f'Test  : {len(test_df):>6,} images')
print()
print('Class distribution per split:')
for name, sdf in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    c = sdf['label'].value_counts(normalize=True).mul(100).round(1)
    print(f'  {name}: {dict(c)}')

In [ ]:
# ── Visualize split sizes ─────────────────────────────────────────────────────
split_counts = {
    'Train': len(train_df),
    'Val':   len(val_df),
    'Test':  len(test_df)
}

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(split_counts.keys(), split_counts.values(),
       color=['#3498db', '#f39c12', '#9b59b6'], edgecolor='white')
for key, val in split_counts.items():
    ax.text(list(split_counts.keys()).index(key), val + 20,
            f'{val:,}', ha='center', fontweight='bold')
ax.set_title('Dataset Split Sizes (70/15/15)')
ax.set_ylabel('Image Count')
plt.tight_layout()
plt.savefig('plots/split_sizes.png', dpi=150)
plt.show()

## 8. Summary & Observations <a id='summary'></a>

In [ ]:
# ── Print a structured EDA summary ────────────────────────────────────────────
print('=' * 60)
print('EDA SUMMARY — Bridge Crack Dataset')
print('=' * 60)
print(f'Total images         : {len(df):,}')
print(f'Classes              : {CLASSES}')
print(f'Imbalance ratio      : {imbalance_ratio:.2f}')
print(f'Dataset mean RGB     : {mean_rgb}')
print(f'Dataset std  RGB     : {std_rgb}')
print()
print('Key observations:')
print('  1. Images are surface-level crack / no-crack binary classification.')
print('  2. Recommend resize to 224×224 for ResNet18/MobileNetV2 compatibility.')
print('  3. Use dataset-specific mean/std (above) for normalization.')
print('  4. Monitor false-alarm rate (FPR on Non-Crack class) as key metric.')
print('  5. Split CSVs saved to data/splits/ for reproducible training.')
print('=' * 60)